# Step 12 — Final Model Revalidation and Deployment Readiness

This notebook performs final revalidation of the frozen Phase 02 Competence → Employability model.

The purpose of this stage is to confirm that:

- The frozen model can be loaded successfully.
- The expected feature schema is preserved.
- Reference predictions can be reproduced.
- Deployment-style survey inputs can be transformed correctly.
- Predictions remain within the valid Employability range.
- Model metadata and integrity information remain consistent.
- The frozen model is ready for integration with the recommendation system and application backend.

No additional model training or hyperparameter tuning is performed in this stage.

In [1]:
from pathlib import Path

import json
import hashlib
import numpy as np
import pandas as pd
import joblib

In [2]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "phase2_competence":
    PROJECT_DIR = PROJECT_DIR.parent.parent

MODEL_DIR = (
    PROJECT_DIR
    / "models"
    / "phase2"
)

RESULT_DIR = (
    PROJECT_DIR
    / "reports"
    / "phase2"
    / "final_revalidation"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_MODEL_PATH = (
    MODEL_DIR
    / "phase2_competence_final_model.pkl"
)

METADATA_PATH = (
    MODEL_DIR
    / "phase2_competence_model_metadata.json"
)

FREEZE_MANIFEST_PATH = (
    MODEL_DIR
    / "phase2_competence_freeze_manifest.json"
)

REFERENCE_PATH = (
    PROJECT_DIR
    / "reports"
    / "phase2"
    / "final_validation"
    / "Reference_Predictions.csv"
)

print("Final model:", FINAL_MODEL_PATH)
print("Metadata:", METADATA_PATH)
print("Freeze manifest:", FREEZE_MANIFEST_PATH)
print("Reference predictions:", REFERENCE_PATH)
print("Revalidation results:", RESULT_DIR)

Final model: d:\internship-employability-project\models\phase2\phase2_competence_final_model.pkl
Metadata: d:\internship-employability-project\models\phase2\phase2_competence_model_metadata.json
Freeze manifest: d:\internship-employability-project\models\phase2\phase2_competence_freeze_manifest.json
Reference predictions: d:\internship-employability-project\reports\phase2\final_validation\Reference_Predictions.csv
Revalidation results: d:\internship-employability-project\reports\phase2\final_revalidation


In [3]:
required_files = {
    "Final Model": FINAL_MODEL_PATH,
    "Model Metadata": METADATA_PATH,
    "Freeze Manifest": FREEZE_MANIFEST_PATH,
    "Reference Predictions": REFERENCE_PATH
}

artifact_check = []

for name, path in required_files.items():

    exists = path.exists()

    artifact_check.append({
        "Artifact": name,
        "Path": str(path),
        "Exists": exists
    })

artifact_check = pd.DataFrame(
    artifact_check
)

artifact_check

,Artifact,Path,Exists
0,Final Model,d:\internship-employability-project\models\pha...,True
1,Model Metadata,d:\internship-employability-project\models\pha...,True
2,Freeze Manifest,d:\internship-employability-project\models\pha...,True
3,Reference Predictions,d:\internship-employability-project\reports\ph...,True


In [4]:

if not artifact_check["Exists"].all():
    raise FileNotFoundError(
        "One or more required Step 11 artifacts are missing."
    )

print("All required artifacts are available.")

All required artifacts are available.


In [5]:
frozen_model = joblib.load(
    FINAL_MODEL_PATH
)

print(
    "Frozen model loaded successfully."
)

print(
    frozen_model
)

Frozen model loaded successfully.
Pipeline(steps=[('scaler', StandardScaler()), ('model', LinearRegression())])


Load Metadata and Manifest

In [6]:
with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:

    metadata = json.load(
        file
    )

with open(
    FREEZE_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as file:

    freeze_manifest = json.load(
        file
    )

print("Model status:", metadata["status"])
print("Model type:", metadata["model_type"])
print("Target:", metadata["target"])
print("Features:", metadata["features"])

Model status: FROZEN
Model type: Linear Regression
Target: EVS_avg
Features: ['LifeCareer_avg', 'LearningInnovation_avg', 'ITMedia_avg']


In [7]:
EXPECTED_FEATURES = [
    "LifeCareer_avg",
    "LearningInnovation_avg",
    "ITMedia_avg"
]

metadata_features = metadata[
    "features"
]

feature_contract_valid = (
    metadata_features
    ==
    EXPECTED_FEATURES
)

print(
    "Expected features:",
    EXPECTED_FEATURES
)

print(
    "Metadata features:",
    metadata_features
)

print(
    "Feature contract valid:",
    feature_contract_valid
)

Expected features: ['LifeCareer_avg', 'LearningInnovation_avg', 'ITMedia_avg']
Metadata features: ['LifeCareer_avg', 'LearningInnovation_avg', 'ITMedia_avg']
Feature contract valid: True


Validate Pipeline Structure

In [8]:
pipeline_steps = list(
    frozen_model.named_steps.keys()
)

expected_pipeline_steps = [
    "scaler",
    "model"
]

pipeline_structure_valid = (
    pipeline_steps
    ==
    expected_pipeline_steps
)

print(
    "Pipeline steps:",
    pipeline_steps
)

print(
    "Pipeline structure valid:",
    pipeline_structure_valid
)

Pipeline steps: ['scaler', 'model']
Pipeline structure valid: True


Load Reference Predictions

In [9]:
reference_df = pd.read_csv(
    REFERENCE_PATH,
    index_col=0
)

print(
    "Reference rows:",
    len(reference_df)
)

reference_df.head()

Reference rows: 10


,LifeCareer_avg,LearningInnovation_avg,ITMedia_avg,Actual_EVS_avg,Predicted_EVS_avg
521,4.0,4.0,4.0,4.333333,4.167467
737,4.0,4.0,4.0,4.333333,4.167467
740,4.0,4.0,4.0,4.000000,4.167467
660,4.0,4.0,4.0,4.250000,4.167467
411,4.0,4.0,4.0,4.333333,4.167467


Reproduce Reference Predictions

In [10]:
X_reference = reference_df[
    EXPECTED_FEATURES
].copy()

expected_predictions = reference_df[
    "Predicted_EVS_avg"
].to_numpy()

revalidated_predictions = frozen_model.predict(
    X_reference
)

prediction_differences = (
    revalidated_predictions
    -
    expected_predictions
)

max_reference_difference = float(
    np.max(
        np.abs(
            prediction_differences
        )
    )
)

reference_predictions_match = bool(
    np.allclose(
        revalidated_predictions,
        expected_predictions,
        rtol=0,
        atol=1e-10
    )
)

print(
    "Reference predictions match:",
    reference_predictions_match
)

print(
    "Maximum reference prediction difference:",
    max_reference_difference
)

Reference predictions match: True
Maximum reference prediction difference: 0.0


Define Survey Items

In [11]:
ITEM_COLUMNS = [
    "C1",
    "C2",
    "C3",
    "C4",
    "C5",
    "C6",
    "C7",
    "C8",
    "C9",
    "C10"
]

Create Input Validation Function

In [12]:
def validate_survey_input(
    survey_response
):

    missing_items = [
        item
        for item in ITEM_COLUMNS
        if item not in survey_response
    ]

    if missing_items:
        raise ValueError(
            f"Missing survey responses: {missing_items}"
        )

    invalid_items = []

    for item in ITEM_COLUMNS:

        value = survey_response[
            item
        ]

        if not isinstance(
            value,
            (int, float)
        ):
            invalid_items.append(
                f"{item}: non-numeric"
            )

        elif value < 1 or value > 5:
            invalid_items.append(
                f"{item}: {value}"
            )

    if invalid_items:
        raise ValueError(
            "Survey values must be numeric and within 1–5. "
            f"Invalid values: {invalid_items}"
        )

    return True

Convert C1–C10 to Final Model Features

In [13]:
def survey_to_construct_features(
    survey_response
):

    validate_survey_input(
        survey_response
    )

    construct_features = {
        "LifeCareer_avg": np.mean([
            survey_response["C1"],
            survey_response["C2"],
            survey_response["C3"],
            survey_response["C4"]
        ]),

        "LearningInnovation_avg": np.mean([
            survey_response["C5"],
            survey_response["C6"],
            survey_response["C7"]
        ]),

        "ITMedia_avg": np.mean([
            survey_response["C8"],
            survey_response["C9"],
            survey_response["C10"]
        ])
    }

    return pd.DataFrame(
        [construct_features],
        columns=EXPECTED_FEATURES
    )

Deployment Prediction Function

In [14]:
def predict_employability(
    survey_response,
    model
):

    model_input = (
        survey_to_construct_features(
            survey_response
        )
    )

    prediction = float(
        model.predict(
            model_input
        )[0]
    )

    return {
        "model_input": (
            model_input
            .iloc[0]
            .to_dict()
        ),

        "predicted_evs_avg": (
            prediction
        ),

        "within_valid_range": bool(
            1 <= prediction <= 5
        )
    }

Test Normal Deployment Profile

In [15]:
deployment_test_profile = {

    "C1": 4,
    "C2": 4,
    "C3": 3,
    "C4": 4,

    "C5": 3,
    "C6": 4,
    "C7": 3,

    "C8": 4,
    "C9": 5,
    "C10": 4
}

deployment_result = (
    predict_employability(
        deployment_test_profile,
        frozen_model
    )
)

deployment_result

{'model_input': {'LifeCareer_avg': 3.75,
  'LearningInnovation_avg': 3.3333333333333335,
  'ITMedia_avg': 4.333333333333333},
 'predicted_evs_avg': 4.054798535255626,
 'within_valid_range': True}

Boundary testing

In [16]:
boundary_profiles = {

    "Minimum Profile": {
        item: 1
        for item in ITEM_COLUMNS
    },

    "Mid Profile": {
        item: 3
        for item in ITEM_COLUMNS
    },

    "Maximum Profile": {
        item: 5
        for item in ITEM_COLUMNS
    }
}

boundary_results = []

for profile_name, survey in (
    boundary_profiles.items()
):

    result = predict_employability(
        survey,
        frozen_model
    )

    boundary_results.append({
        "Profile": profile_name,
        **result["model_input"],
        "Predicted_EVS_avg": (
            result[
                "predicted_evs_avg"
            ]
        ),
        "Within_Valid_Range": (
            result[
                "within_valid_range"
            ]
        )
    })

boundary_results = pd.DataFrame(
    boundary_results
)

boundary_results

,Profile,LifeCareer_avg,LearningInnovation_avg,ITMedia_avg,Predicted_EVS_avg,Within_Valid_Range
0,Minimum Profile,1.0,1.0,1.0,1.823072,True
1,Mid Profile,3.0,3.0,3.0,3.386002,True
2,Maximum Profile,5.0,5.0,5.0,4.948932,True


Validate Ordering

In [17]:
low_prediction = boundary_results.loc[
    boundary_results["Profile"]
    == "Minimum Profile",
    "Predicted_EVS_avg"
].iloc[0]

mid_prediction = boundary_results.loc[
    boundary_results["Profile"]
    == "Mid Profile",
    "Predicted_EVS_avg"
].iloc[0]

high_prediction = boundary_results.loc[
    boundary_results["Profile"]
    == "Maximum Profile",
    "Predicted_EVS_avg"
].iloc[0]

logical_ordering_valid = bool(
    low_prediction
    <
    mid_prediction
    <
    high_prediction
)

print(
    "Logical prediction ordering valid:",
    logical_ordering_valid
)

Logical prediction ordering valid: True


Invalid input testing

In [18]:
invalid_missing_profile = {
    "C1": 4,
    "C2": 4
}

try:

    predict_employability(
        invalid_missing_profile,
        frozen_model
    )

    missing_input_validation_passed = False

except ValueError as error:

    missing_input_validation_passed = True

    print(
        "Missing-input validation correctly rejected input."
    )

    print(
        error
    )

Missing-input validation correctly rejected input.
Missing survey responses: ['C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10']


Out-of-Range Input Test

In [19]:
invalid_range_profile = {

    "C1": 6,
    "C2": 4,
    "C3": 3,
    "C4": 4,

    "C5": 3,
    "C6": 4,
    "C7": 3,

    "C8": 4,
    "C9": 5,
    "C10": 4
}

try:

    predict_employability(
        invalid_range_profile,
        frozen_model
    )

    range_input_validation_passed = False

except ValueError as error:

    range_input_validation_passed = True

    print(
        "Range validation correctly rejected input."
    )

    print(
        error
    )

Range validation correctly rejected input.
Survey values must be numeric and within 1–5. Invalid values: ['C1: 6']


Artifact integrity
Cell 20 — SHA-256 Function

In [20]:
def calculate_sha256(
    file_path
):

    sha256 = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file:

        while True:

            chunk = file.read(
                8192
            )

            if not chunk:
                break

            sha256.update(
                chunk
            )

    return (
        sha256.hexdigest()
    )

Verify Frozen Model Hash

In [21]:
current_model_hash = (
    calculate_sha256(
        FINAL_MODEL_PATH
    )
)

expected_model_hash = (
    freeze_manifest[
        "model_sha256"
    ]
)

hash_matches = bool(
    current_model_hash
    ==
    expected_model_hash
)

print(
    "Current model hash:",
    current_model_hash
)

print(
    "Expected frozen hash:",
    expected_model_hash
)

print(
    "Model integrity valid:",
    hash_matches
)

Current model hash: 4d5feaff2047c176f1b1387d40a4c9f248ddcb5aaac7a241c3496654a13a58b8
Expected frozen hash: 4d5feaff2047c176f1b1387d40a4c9f248ddcb5aaac7a241c3496654a13a58b8
Model integrity valid: True


Metadata consistency
Cell 22 — Metadata Consistency Check

In [22]:
metadata_checks = {

    "Frozen_Status": (
        metadata.get(
            "status"
        )
        ==
        "FROZEN"
    ),

    "Correct_Target": (
        metadata.get(
            "target"
        )
        ==
        "EVS_avg"
    ),

    "Correct_Features": (
        metadata.get(
            "features"
        )
        ==
        EXPECTED_FEATURES
    ),

    "Correct_Model_Type": (
        metadata.get(
            "model_type"
        )
        ==
        "Linear Regression"
    )
}

metadata_check_df = pd.DataFrame({
    "Check": (
        metadata_checks.keys()
    ),

    "Passed": (
        metadata_checks.values()
    )
})

metadata_check_df

,Check,Passed
0,Frozen_Status,True
1,Correct_Target,True
2,Correct_Features,True
3,Correct_Model_Type,True


Recommendation Input Compatibility

In [23]:
recommendation_compatible_input = (
    survey_to_construct_features(
        deployment_test_profile
    )
)

recommendation_compatible_input

,LifeCareer_avg,LearningInnovation_avg,ITMedia_avg
0,3.75,3.333333,4.333333


In [24]:
recommendation_input_compatible = bool(

    recommendation_compatible_input
    .columns
    .tolist()

    ==
    EXPECTED_FEATURES
)

print(
    "Recommendation/model feature compatibility:",
    recommendation_input_compatible
)

Recommendation/model feature compatibility: True


Build deployment package metadata
Cell 24 — Create Deployment Contract

In [25]:
deployment_contract = {

    "phase": (
        "Phase 02"
    ),

    "purpose": (
        "Competence to Employability prediction"
    ),

    "input_items": (
        ITEM_COLUMNS
    ),

    "derived_features": (
        EXPECTED_FEATURES
    ),

    "target": (
        "EVS_avg"
    ),

    "valid_input_range": {
        "minimum": 1,
        "maximum": 5
    },

    "pipeline": [
        "StandardScaler",
        "LinearRegression"
    ],

    "model_file": (
        FINAL_MODEL_PATH.name
    ),

    "model_hash": (
        current_model_hash
    ),

    "recommendation_layer": (
        "SHAP-grounded recommendation system"
    ),

    "model_status": (
        "FROZEN"
    )
}

DEPLOYMENT_CONTRACT_PATH = (
    MODEL_DIR
    / "phase2_competence_deployment_contract.json"
)

with open(
    DEPLOYMENT_CONTRACT_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        deployment_contract,
        file,
        indent=4
    )

print(
    "Deployment contract saved:",
    DEPLOYMENT_CONTRACT_PATH
)

Deployment contract saved: d:\internship-employability-project\models\phase2\phase2_competence_deployment_contract.json


Final revalidation summary

In [26]:
final_revalidation_summary = pd.DataFrame({

    "Revalidation_Check": [

        "Required artifacts available",

        "Frozen model loaded",

        "Feature contract",

        "Pipeline structure",

        "Reference prediction reproduction",

        "Prediction range",

        "Logical boundary ordering",

        "Missing-input rejection",

        "Out-of-range rejection",

        "Model hash integrity",

        "Metadata consistency",

        "Recommendation compatibility"
    ],

    "Status": [

        (
            "PASS"
            if artifact_check[
                "Exists"
            ].all()
            else "FAIL"
        ),

        "PASS",

        (
            "PASS"
            if feature_contract_valid
            else "FAIL"
        ),

        (
            "PASS"
            if pipeline_structure_valid
            else "FAIL"
        ),

        (
            "PASS"
            if reference_predictions_match
            else "FAIL"
        ),

        (
            "PASS"
            if deployment_result[
                "within_valid_range"
            ]
            else "REVIEW"
        ),

        (
            "PASS"
            if logical_ordering_valid
            else "FAIL"
        ),

        (
            "PASS"
            if missing_input_validation_passed
            else "FAIL"
        ),

        (
            "PASS"
            if range_input_validation_passed
            else "FAIL"
        ),

        (
            "PASS"
            if hash_matches
            else "FAIL"
        ),

        (
            "PASS"
            if all(
                metadata_checks.values()
            )
            else "FAIL"
        ),

        (
            "PASS"
            if recommendation_input_compatible
            else "FAIL"
        )
    ]
})

final_revalidation_summary

,Revalidation_Check,Status
0,Required artifacts available,PASS
1,Frozen model loaded,PASS
2,Feature contract,PASS
3,Pipeline structure,PASS
4,Reference prediction reproduction,PASS
5,Prediction range,PASS
6,Logical boundary ordering,PASS
7,Missing-input rejection,PASS
8,Out-of-range rejection,PASS
9,Model hash integrity,PASS


Save Revalidation Outputs

In [27]:
artifact_check.to_csv(
    RESULT_DIR
    / "Artifact_Availability_Check.csv",
    index=False
)

boundary_results.to_csv(
    RESULT_DIR
    / "Deployment_Boundary_Test.csv",
    index=False
)

metadata_check_df.to_csv(
    RESULT_DIR
    / "Metadata_Consistency_Check.csv",
    index=False
)

final_revalidation_summary.to_csv(
    RESULT_DIR
    / "Final_Revalidation_Summary.csv",
    index=False
)

print(
    "Step 12 revalidation outputs saved."
)

Step 12 revalidation outputs saved.


Create Final Revalidation JSON

In [28]:
revalidation_report = {

    "model_file": (
        FINAL_MODEL_PATH.name
    ),

    "model_hash_valid": (
        hash_matches
    ),

    "reference_predictions_match": (
        reference_predictions_match
    ),

    "maximum_reference_prediction_difference": (
        max_reference_difference
    ),

    "feature_contract_valid": (
        feature_contract_valid
    ),

    "pipeline_structure_valid": (
        pipeline_structure_valid
    ),

    "deployment_prediction": (
        deployment_result[
            "predicted_evs_avg"
        ]
    ),

    "deployment_prediction_in_range": (
        deployment_result[
            "within_valid_range"
        ]
    ),

    "logical_boundary_ordering": (
        logical_ordering_valid
    ),

    "invalid_input_checks": {

        "missing_input_rejected": (
            missing_input_validation_passed
        ),

        "out_of_range_input_rejected": (
            range_input_validation_passed
        )
    },

    "recommendation_compatibility": (
        recommendation_input_compatible
    ),

    "final_status": (
        "DEPLOYMENT_READY"
        if (
            final_revalidation_summary[
                "Status"
            ]
            ==
            "PASS"
        ).all()
        else "REVIEW_REQUIRED"
    )
}

REVALIDATION_REPORT_PATH = (
    RESULT_DIR
    / "Final_Revalidation_Report.json"
)

with open(
    REVALIDATION_REPORT_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        revalidation_report,
        file,
        indent=4
    )

revalidation_report

{'model_file': 'phase2_competence_final_model.pkl',
 'model_hash_valid': True,
 'reference_predictions_match': True,
 'maximum_reference_prediction_difference': 0.0,
 'feature_contract_valid': True,
 'pipeline_structure_valid': True,
 'deployment_prediction': 4.054798535255626,
 'deployment_prediction_in_range': True,
 'logical_boundary_ordering': True,
 'invalid_input_checks': {'missing_input_rejected': True,
  'out_of_range_input_rejected': True},
 'recommendation_compatibility': True,
 'final_status': 'DEPLOYMENT_READY'}

In [29]:
FINAL_STATUS = (
    revalidation_report[
        "final_status"
    ]
)

print(
    "FINAL PHASE 02 MODEL STATUS:",
    FINAL_STATUS
)

FINAL PHASE 02 MODEL STATUS: DEPLOYMENT_READY


## Step 12 Conclusion

The frozen Phase 02 Competence → Employability model was successfully revalidated without additional training or hyperparameter tuning.

The saved model artifact was loaded successfully, the expected feature and pipeline contracts were preserved, and reference predictions were reproduced within the defined numerical tolerance.

Deployment-style C1–C10 survey inputs were validated, transformed into the three competence constructs, and processed successfully by the frozen prediction pipeline. Boundary and invalid-input tests confirmed appropriate input handling and logical model behaviour.

The frozen model hash matched the Step 11 integrity manifest, confirming that the model artifact had not changed after freezing. Model metadata and recommendation-system feature compatibility were also successfully verified.

Therefore, the final Phase 02 Competence → Employability model is considered deployment-ready for integration with the application and recommendation system.

## Final Revalidation Result

The frozen Phase 02 Competence → Employability model successfully passed all final revalidation and deployment-readiness checks.

The saved model artifact was loaded without retraining, and all stored reference predictions were reproduced exactly, with a maximum prediction difference of 0.0. The required feature schema, preprocessing pipeline, model metadata, and artifact hash were also successfully verified.

Deployment-style survey inputs were correctly transformed from C1–C10 responses into the three final competence constructs: Life & Career Skills, Learning & Innovation Skills, and IT & Media Skills. Boundary testing demonstrated logically ordered predictions, while invalid and incomplete inputs were correctly rejected.

The model was also confirmed to remain compatible with the recommendation system. Therefore, the final Phase 02 Competence → Employability model is considered deployment-ready.

Determinism Test

Sinhala meaning: same input එකට every time same output එනවාද බලනවා.

In [32]:
test_input = deployment_test_profile

predictions = []

for _ in range(10):
    result = predict_employability(
        test_input,
        frozen_model
    )
    predictions.append(
        result["predicted_evs_avg"]
    )

deterministic_check = (
    max(predictions)
    -
    min(predictions)
) == 0

print(
    "Deterministic prediction:",
    deterministic_check
)

print(
    predictions
)

Deterministic prediction: True
[4.054798535255626, 4.054798535255626, 4.054798535255626, 4.054798535255626, 4.054798535255626, 4.054798535255626, 4.054798535255626, 4.054798535255626, 4.054798535255626, 4.054798535255626]


Random Valid Profile Stress Test

In [33]:
np.random.seed(42)

stress_test_results = []

for i in range(100):

    random_profile = {
        item: int(
            np.random.randint(1, 6)
        )
        for item in ITEM_COLUMNS
    }

    result = predict_employability(
        random_profile,
        frozen_model
    )

    stress_test_results.append({
        "Test_ID": i + 1,
        "Prediction": result[
            "predicted_evs_avg"
        ],
        "Within_Valid_Range": result[
            "within_valid_range"
        ]
    })

stress_test_df = pd.DataFrame(
    stress_test_results
)

stress_test_df.head()

,Test_ID,Prediction,Within_Valid_Range
0,1,3.953877,True
1,2,3.683194,True
2,3,3.328389,True
3,4,3.443614,True
4,5,3.215721,True


In [34]:
print(
    "Total tests:",
    len(stress_test_df)
)

print(
    "Valid predictions:",
    stress_test_df[
        "Within_Valid_Range"
    ].sum()
)

print(
    "Invalid predictions:",
    (
        ~stress_test_df[
            "Within_Valid_Range"
        ]
    ).sum()
)

Total tests: 100
Valid predictions: 100
Invalid predictions: 0


In [35]:
wrong_order_input = pd.DataFrame([{
    "ITMedia_avg": 4.0,
    "LifeCareer_avg": 3.5,
    "LearningInnovation_avg": 4.2
}])

correct_order = (
    wrong_order_input.columns.tolist()
    ==
    EXPECTED_FEATURES
)

print(
    "Correct feature order:",
    correct_order
)

Correct feature order: False


In [36]:
wrong_order_input = wrong_order_input[
    EXPECTED_FEATURES
]

මේකෙන් order always controlled.

Final Deployment Checklist

In [37]:
deployment_readiness_checks = {
    "Artifacts available": True,
    "Model hash valid": hash_matches,
    "Reference predictions reproducible": reference_predictions_match,
    "Feature contract valid": feature_contract_valid,
    "Pipeline structure valid": pipeline_structure_valid,
    "Invalid inputs rejected": (
        missing_input_validation_passed
        and range_input_validation_passed
    ),
    "Logical boundary behaviour": logical_ordering_valid,
    "Recommendation compatible": recommendation_input_compatible,
    "Deterministic predictions": deterministic_check
}

deployment_readiness_df = pd.DataFrame({
    "Check": deployment_readiness_checks.keys(),
    "Passed": deployment_readiness_checks.values()
})

deployment_readiness_df

,Check,Passed
0,Artifacts available,True
1,Model hash valid,True
2,Reference predictions reproducible,True
3,Feature contract valid,True
4,Pipeline structure valid,True
5,Invalid inputs rejected,True
6,Logical boundary behaviour,True
7,Recommendation compatible,True
8,Deterministic predictions,True
